In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def entropy(x, b):
    s = 0
    for p in x:
        if p > 0:
            s += -p * math.log(p, b)
    return s

def visualize_regions_keras(model, MAX, nGrid):
    a1 = np.linspace(-MAX, MAX, nGrid)
    a2 = np.linspace(-MAX, MAX, nGrid)
    A1, A2 = np.meshgrid(a1, a2)
    A1 = A1.flatten()
    A2 = A2.flatten()
    A = np.vstack((A1, A2)).T
    B = model.predict(A, verbose=0).argmax(axis=1)
    B = B.reshape(nGrid, nGrid)
    B = np.flipud(B)
    prob_2D = model.predict(A, verbose=0)
    S = np.array([entropy(p, 2) for p in prob_2D])
    S = S.reshape(nGrid, nGrid)
    S = np.flipud(S)
    return B, S

# Scenario 1

Baseline model. Experiment with number of layers, neurons per layer, activation functions, batch size, and epochs.

In [ ]:
from sklearn.datasets import make_classification, make_blobs, make_moons, make_circles
from sklearn.model_selection import train_test_split

n_samples = 1000
choice = 3

if   choice == 0:
    X, y = make_blobs(n_samples=n_samples, n_features=2, centers=2, cluster_std=5, random_state=10)
elif choice == 1:
    X, y = make_classification(n_samples=n_samples, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1, n_classes=3, random_state=41)
elif choice == 2:
    X, y = make_moons(n_samples=n_samples, noise=0.2, random_state=42)
elif choice == 3:
    X, y = make_circles(n_samples=n_samples, noise=0.1, factor=0.5, random_state=42)

X = X - X.mean(axis=0)
MAX = np.max(np.abs(X))
n_classes = np.unique(y).shape[0]

plt.scatter(X[:,0], X[:,1], c=y, s=10, cmap='RdBu')
plt.xlim(-MAX, MAX); plt.ylim(-MAX, MAX)
plt.gca().set_aspect('equal')
plt.title('Dataset')
plt.show()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
from keras.models import Sequential
from keras.layers import *
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

model = Sequential([
    Input(shape=X.shape[1:]),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(n_classes, activation='softmax'),
])

model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train, y_train,
    epochs=300,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test, y_test),
    verbose=0
)

y_test_pred = model.predict(X_test).argmax(axis=1)
B, S = visualize_regions_keras(model, MAX, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epochs'); axes[0].legend()

extent = [-MAX, MAX, -MAX, MAX]
axes[1].imshow(B, interpolation='none', extent=extent, cmap='RdBu', alpha=0.4)
axes[1].scatter(X_test[:,0], X_test[:,1], c=y_test, s=10, cmap='RdBu')
axes[1].set_aspect('equal'); axes[1].set_title('Decision Boundary (Test)')

plt.tight_layout(); plt.show()
print(classification_report(y_test, y_test_pred))

# Scenario 2: Feature scaling

Unlike regression, classification labels are integers and require no scaling. We do benefit from scaling the **features** with `StandardScaler` so that all inputs have zero mean and unit variance — this helps gradient descent converge faster and more reliably.

In [ ]:
from sklearn.preprocessing import StandardScaler

n_samples = 1000
choice = 3

if   choice == 0:
    X, y = make_blobs(n_samples=n_samples, n_features=2, centers=2, cluster_std=5, random_state=10)
elif choice == 1:
    X, y = make_classification(n_samples=n_samples, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1, n_classes=3, random_state=41)
elif choice == 2:
    X, y = make_moons(n_samples=n_samples, noise=0.2, random_state=42)
elif choice == 3:
    X, y = make_circles(n_samples=n_samples, noise=0.1, factor=0.5, random_state=42)

X = X - X.mean(axis=0)
MAX = np.max(np.abs(X))
n_classes = np.unique(y).shape[0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

plt.scatter(X_train_s[:,0], X_train_s[:,1], c=y_train, s=10, cmap='RdBu')
plt.gca().set_aspect('equal')
plt.title('Scaled Training Features')
plt.show()

In [ ]:
model = Sequential([
    Input(shape=X.shape[1:]),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(n_classes, activation='softmax'),
])

model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train_s, y_train,
    epochs=100,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test_s, y_test),
    verbose=0
)

y_test_pred = model.predict(X_test_s).argmax(axis=1)

# For visualization, rebuild grid in scaled space
MAX_S = np.max(np.abs(X_train_s))
B, S = visualize_regions_keras(model, MAX_S, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epochs'); axes[0].legend()

extent_s = [-MAX_S, MAX_S, -MAX_S, MAX_S]
axes[1].imshow(B, interpolation='none', extent=extent_s, cmap='RdBu', alpha=0.4)
axes[1].scatter(X_test_s[:,0], X_test_s[:,1], c=y_test, s=10, cmap='RdBu')
axes[1].set_aspect('equal'); axes[1].set_title('Decision Boundary (Test, Scaled)')

plt.tight_layout(); plt.show()
print(classification_report(y_test, y_test_pred))

# Scenario 3: Multi-class problem

Switch to a 3-class dataset (`make_classification` with `n_classes=3`). The model output layer automatically adjusts to `n_classes` units. We continue using feature scaling and `sparse_categorical_crossentropy`.

In [ ]:
n_samples = 1000
choice = 1  # make_classification, 3 classes

if   choice == 0:
    X, y = make_blobs(n_samples=n_samples, n_features=2, centers=2, cluster_std=5, random_state=10)
elif choice == 1:
    X, y = make_classification(n_samples=n_samples, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1, n_classes=3, random_state=41)
elif choice == 2:
    X, y = make_moons(n_samples=n_samples, noise=0.2, random_state=42)
elif choice == 3:
    X, y = make_circles(n_samples=n_samples, noise=0.1, factor=0.5, random_state=42)

X = X - X.mean(axis=0)
MAX = np.max(np.abs(X))
n_classes = np.unique(y).shape[0]

plt.scatter(X[:,0], X[:,1], c=y, s=10, cmap='RdBu')
plt.xlim(-MAX, MAX); plt.ylim(-MAX, MAX)
plt.gca().set_aspect('equal')
plt.title(f'Dataset ({n_classes} classes)')
plt.show()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
model = Sequential([
    Input(shape=X.shape[1:]),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(n_classes, activation='softmax'),
])

model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train_s, y_train,
    epochs=100,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test_s, y_test),
    verbose=0
)

y_test_pred = model.predict(X_test_s).argmax(axis=1)
MAX_S = np.max(np.abs(X_train_s))
B, S = visualize_regions_keras(model, MAX_S, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epochs'); axes[0].legend()

extent_s = [-MAX_S, MAX_S, -MAX_S, MAX_S]
axes[1].imshow(B, interpolation='none', extent=extent_s, cmap='RdBu', alpha=0.4)
axes[1].scatter(X_test_s[:,0], X_test_s[:,1], c=y_test, s=10, cmap='RdBu')
axes[1].set_aspect('equal'); axes[1].set_title('Decision Boundary (3-class, Test)')

plt.tight_layout(); plt.show()
print(classification_report(y_test, y_test_pred))

# Scenario 4: Change learning rate and optimizer

The default Adam learning rate is `0.001`. A higher rate (e.g., `0.01`) can converge faster but may overshoot. We keep the 3-class dataset and feature scaling from Scenario 3.

In [ ]:
n_samples = 1000
choice = 1

if   choice == 0:
    X, y = make_blobs(n_samples=n_samples, n_features=2, centers=2, cluster_std=5, random_state=10)
elif choice == 1:
    X, y = make_classification(n_samples=n_samples, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1, n_classes=3, random_state=41)
elif choice == 2:
    X, y = make_moons(n_samples=n_samples, noise=0.2, random_state=42)
elif choice == 3:
    X, y = make_circles(n_samples=n_samples, noise=0.1, factor=0.5, random_state=42)

X = X - X.mean(axis=0)
MAX = np.max(np.abs(X))
n_classes = np.unique(y).shape[0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
model = Sequential([
    Input(shape=X.shape[1:]),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(10, activation='relu'),
    Dense(n_classes, activation='softmax'),
])

optimizer = Adam(learning_rate=0.01)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train_s, y_train,
    epochs=100,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test_s, y_test),
    verbose=0
)

y_test_pred = model.predict(X_test_s).argmax(axis=1)
MAX_S = np.max(np.abs(X_train_s))
B, S = visualize_regions_keras(model, MAX_S, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy (lr=0.01)'); axes[0].set_xlabel('Epochs'); axes[0].legend()

extent_s = [-MAX_S, MAX_S, -MAX_S, MAX_S]
axes[1].imshow(B, interpolation='none', extent=extent_s, cmap='RdBu', alpha=0.4)
axes[1].scatter(X_test_s[:,0], X_test_s[:,1], c=y_test, s=10, cmap='RdBu')
axes[1].set_aspect('equal'); axes[1].set_title('Decision Boundary (Test)')

plt.tight_layout(); plt.show()
print(classification_report(y_test, y_test_pred))

# Scenario 5: Including dropout

**Dropout** randomly sets a fraction of neuron activations to zero during each training step. This prevents co-adaptation of neurons and acts as a form of regularization, reducing overfitting. We also increase neurons to 20 per layer to give the network more capacity before applying regularization.

In [ ]:
n_samples = 1000
choice = 1

if   choice == 0:
    X, y = make_blobs(n_samples=n_samples, n_features=2, centers=2, cluster_std=5, random_state=10)
elif choice == 1:
    X, y = make_classification(n_samples=n_samples, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1, n_classes=3, random_state=41)
elif choice == 2:
    X, y = make_moons(n_samples=n_samples, noise=0.2, random_state=42)
elif choice == 3:
    X, y = make_circles(n_samples=n_samples, noise=0.1, factor=0.5, random_state=42)

X = X - X.mean(axis=0)
MAX = np.max(np.abs(X))
n_classes = np.unique(y).shape[0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
model = Sequential([
    Input(shape=X.shape[1:]),
    Dense(20, activation='relu'),
    Dropout(0.25),
    Dense(20, activation='relu'),
    Dropout(0.25),
    Dense(20, activation='relu'),
    Dense(n_classes, activation='softmax'),
])

optimizer = Adam(learning_rate=0.01)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train_s, y_train,
    epochs=100,
    batch_size=64,
    shuffle=True,
    validation_data=(X_test_s, y_test),
    verbose=0
)

y_test_pred = model.predict(X_test_s).argmax(axis=1)
MAX_S = np.max(np.abs(X_train_s))
B, S = visualize_regions_keras(model, MAX_S, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy (Dropout=0.25)'); axes[0].set_xlabel('Epochs'); axes[0].legend()

extent_s = [-MAX_S, MAX_S, -MAX_S, MAX_S]
axes[1].imshow(B, interpolation='none', extent=extent_s, cmap='RdBu', alpha=0.4)
axes[1].scatter(X_test_s[:,0], X_test_s[:,1], c=y_test, s=10, cmap='RdBu')
axes[1].set_aspect('equal'); axes[1].set_title('Decision Boundary (Test)')

plt.tight_layout(); plt.show()
print(classification_report(y_test, y_test_pred))

# Scenario 6: Add batch normalization

**Batch Normalization** normalizes the activations of each layer across the mini-batch, stabilizing training and often allowing higher learning rates. It is placed **after** the linear transformation and **before** (or sometimes after) the activation. We use it here with reduced dropout (`0.1`).

In [ ]:
n_samples = 1000
choice = 1

if   choice == 0:
    X, y = make_blobs(n_samples=n_samples, n_features=2, centers=2, cluster_std=5, random_state=10)
elif choice == 1:
    X, y = make_classification(n_samples=n_samples, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1, n_classes=3, random_state=41)
elif choice == 2:
    X, y = make_moons(n_samples=n_samples, noise=0.2, random_state=42)
elif choice == 3:
    X, y = make_circles(n_samples=n_samples, noise=0.1, factor=0.5, random_state=42)

X = X - X.mean(axis=0)
MAX = np.max(np.abs(X))
n_classes = np.unique(y).shape[0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
model = Sequential([
    Input(shape=X.shape[1:]),
    Dense(20),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.1),
    Dense(20),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.1),
    Dense(20, activation='relu'),
    Dense(n_classes, activation='softmax'),
])

optimizer = Adam(learning_rate=0.01)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train_s, y_train,
    epochs=100,
    batch_size=64,
    shuffle=True,
    validation_data=(X_test_s, y_test),
    verbose=0
)

y_test_pred = model.predict(X_test_s).argmax(axis=1)
MAX_S = np.max(np.abs(X_train_s))
B, S = visualize_regions_keras(model, MAX_S, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy (BatchNorm + Dropout=0.1)'); axes[0].set_xlabel('Epochs'); axes[0].legend()

extent_s = [-MAX_S, MAX_S, -MAX_S, MAX_S]
axes[1].imshow(B, interpolation='none', extent=extent_s, cmap='RdBu', alpha=0.4)
axes[1].scatter(X_test_s[:,0], X_test_s[:,1], c=y_test, s=10, cmap='RdBu')
axes[1].set_aspect('equal'); axes[1].set_title('Decision Boundary (Test)')

plt.tight_layout(); plt.show()
print(classification_report(y_test, y_test_pred))